In [1]:
from google.colab import files
uploaded = files.upload()
print(list(uploaded.keys()))

Saving answer_key.xlsx to answer_key (1).xlsx
Saving sample_POs.txt to sample_POs (1).txt
['answer_key (1).xlsx', 'sample_POs (1).txt']


In [2]:
text = open("sample_POs.txt", encoding="utf-8-sig").read()
print(len(text), "characters")
print("Number of POs found:", text.count("From:"))
print(text[:300])

4785 characters
Number of POs found: 8
PO 1 (formal) 


From: procurement@nusantaraniaga.example
To: orders@sunbirdgoods.example
Date: 2 July 2026
Subject: Purchase Order NN-PO-2026-0701


Dear Sunbird Consumer Goods Sales Team,


Please accept this purchase order from Nusantara Niaga PT.


Purchase Order No.: NN-PO-2026-0701
Order Date:


In [3]:
import pandas as pd
key = pd.read_excel("answer_key.xlsx", sheet_name="Answer key")
print(key.shape)
print(key.columns.tolist())
print(key.head())

(28, 11)
['PO number', 'PO date', 'Sender email', 'Distributor', 'Product', 'Quantity', 'Expected Quantity', 'Unit price', 'Line total', 'Delivery date', 'Errors I should expect my tool to catch']
         PO number    PO date                        Sender email  \
0  NN-PO-2026-0701 2026-07-02  procurement@nusantaraniaga.example   
1  NN-PO-2026-0701 2026-07-02  procurement@nusantaraniaga.example   
2  NN-PO-2026-0701 2026-07-02  procurement@nusantaraniaga.example   
3    KVD/2026/0315 2026-07-09      orders@klangvalleydist.example   
4    KVD/2026/0315 2026-07-09      orders@klangvalleydist.example   

                 Distributor                    Product Quantity  \
0         Nusantara Niaga PT       Instant Kopi Mix 20s      500   
1         Nusantara Niaga PT  Sambal Chilli Sauce 250ml      300   
2         Nusantara Niaga PT      Coconut Biscuits 200g      800   
3  Klang Valley Distribution    Electrolyte Sachets 12s      400   
4  Klang Valley Distribution      Vitamin C Gumm

In [4]:
import re

parts = re.split(r"(?m)^\s*PO (\d+) \(([^)]*)\)\s*$", text)

pos = []
for i in range(1, len(parts), 3):
    pos.append({
        "po_no": int(parts[i]),
        "style": parts[i + 1],
        "email": parts[i + 2].strip()
    })

print("POs found:", len(pos))
for p in pos:
    print(p["po_no"], "|", p["style"], "|", len(p["email"]), "characters")

POs found: 8
1 | formal | 647 characters
2 | messy | 464 characters
3 | table made of text | 811 characters
4 | casual, abbreviations | 443 characters
5 | formal | 620 characters
6 | short, semi-formal | 450 characters
7 | messy | 490 characters
8 | neat, structured | 648 characters


In [5]:
print(pos[3]["email"])

From: carlo@manilabaytrading.example
To: orders@sunbirdgoods.example
Date: 3 Aug 2026
Subject: Aug order pls


Hi team!


Manila Bay Trading here. Pls see our Aug restock below, no PO# yet
from accounting, will send later.


- Ultra Laundry Detergent 2kg - 240 pcs @ 9.30
- Fabric Softener 1.5L - two hundred pcs @ 5.53
- Lavender Floor Cleaner 1L - 150 pcs @ 4.20


Pls del by 31 Aug 2026 to our Subic WH. Prices in SGD as usual.
Thx!


Carlo


In [6]:
import json

raw = '''{“sender_email”:”carlo@manilabaytrading.example”,”distributor_as_written”:”Manila Bay Trading”,”po_number_subject”:null,”po_number_body”:null,”po_date”:”2026-08-03",”delivery_date”:”2026-08-31",”currency”:”SGD”,”items”:[{“product”:”Ultra Laundry Detergent 2kg”,”quantity_raw”:”240 pcs”,”quantity_number”:240,”unit_price”:9.30},{“product”:”Fabric Softener 1.5L”,”quantity_raw”:”two hundred pcs”,”quantity_number”:null,”unit_price”:5.53},{“product”:”Lavender Floor Cleaner 1L”,”quantity_raw”:”150 pcs”,”quantity_number”:150,”unit_price”:4.20}]}'''
raw = raw.replace("“", '"').replace("”", '"')

data = json.loads(raw)
print("Sender:", data["sender_email"])
print("PO date:", data["po_date"], "| Delivery:", data["delivery_date"])
print("Number of items:", len(data["items"]))
for item in data["items"]:
    print(item["product"], "|", item["quantity_raw"], "|", item["unit_price"])

Sender: carlo@manilabaytrading.example
PO date: 2026-08-03 | Delivery: 2026-08-31
Number of items: 3
Ultra Laundry Detergent 2kg | 240 pcs | 9.3
Fabric Softener 1.5L | two hundred pcs | 5.53
Lavender Floor Cleaner 1L | 150 pcs | 4.2


In [7]:
import pandas as pd

In [8]:
from google.colab import files
uploaded = files.upload()
print(list(uploaded.keys()))

Saving answer_key.xlsx to answer_key (2).xlsx
Saving sample_POs.txt to sample_POs (2).txt
['answer_key (2).xlsx', 'sample_POs (2).txt']


In [9]:
import re
import json
import pandas as pd

# 1. Read the POs and split them
text = open("sample_POs.txt", encoding="utf-8-sig").read()
parts = re.split(r"(?m)^\s*PO (\d+) \(([^)]*)\)\s*$", text)
pos = []
for i in range(1, len(parts), 3):
    pos.append({
        "po_no": int(parts[i]),
        "style": parts[i + 1],
        "email": parts[i + 2].strip()
    })

# 2. Read the answer key and the registry
key = pd.read_excel("answer_key.xlsx", sheet_name="Answer key")
registry = pd.read_excel("answer_key.xlsx", sheet_name="Distributor registry")
registry = registry.dropna(subset=["Country"])

# 3. Standard price list
standard_prices = {
    "Instant Kopi Mix 20s": 6.06,
    "Sambal Chilli Sauce 250ml": 3.34,
    "Jasmine Green Tea 25 bags": 3.70,
    "Coconut Biscuits 200g": 2.10,
    "Electrolyte Sachets 12s": 6.83,
    "Vitamin C Gummies 60s": 11.35,
    "Collagen Drink Box 10s": 18.56,
    "Herbal Throat Lozenges 24s": 3.28,
    "Ultra Laundry Detergent 2kg": 9.30,
    "Fabric Softener 1.5L": 5.53,
    "Citrus Dishwash Liquid 750ml": 2.98,
    "Lavender Floor Cleaner 1L": 4.20,
    "Aloe Fresh Body Wash 500ml": 4.04,
    "Gentle Sunscreen SPF50 100ml": 8.58,
    "Herbal Shampoo 400ml": 4.77,
    "Whitening Toothpaste 150g": 2.49,
}

# 4. Quick checks
print("POs found:", len(pos))
print("Answer key shape:", key.shape)
print("Distributors:", len(registry))
print("Products:", len(standard_prices))
print()
print(registry)

POs found: 8
Answer key shape: (28, 11)
Distributors: 8
Products: 16

                     Distributor      Country  \
0             Nusantara Niaga PT    Indonesia   
1  Jakarta Prima Distribution PT    Indonesia   
2      Klang Valley Distribution     Malaysia   
3       Penang Consumer Supplies     Malaysia   
4    Siam Retail Partners Co Ltd     Thailand   
5        Manila Bay Trading Corp  Philippines   
6  Saigon Fresh Distribution JSC      Vietnam   
7      Lion City Trading Pte Ltd    Singapore   

                 Official ordering email  
0     procurement@nusantaraniaga.example  
1            gudang@jakartaprima.example  
2         orders@klangvalleydist.example  
3          orders@penangconsumer.example  
4  purchasing@siamretailpartners.example  
5         carlo@manilabaytrading.example  
6         purchasing@saigonfresh.example  
7            ops@lioncitytrading.example  


In [10]:
from datetime import date

def validate(data):
    errors = []

    # Check 1: missing PO number
    if not data["po_number_subject"] and not data["po_number_body"]:
        errors.append("Missing PO number")

    # Check 5: delivery date before order date
    if data["po_date"] and data["delivery_date"]:
        po_date = date.fromisoformat(data["po_date"])
        delivery = date.fromisoformat(data["delivery_date"])
        if delivery < po_date:
            errors.append("Delivery date before order date")

    # Checks 2, 3, 4: look at each item
    for item in data["items"]:
        name = item["product"]

        if item["quantity_number"] is None:
            errors.append(f"Quantity not a number: {name} ('{item['quantity_raw']}')")

        if name not in standard_prices:
            errors.append(f"Unknown product: {name}")
        else:
            std = standard_prices[name]
            price = item["unit_price"]
            if price is not None and abs(price - std) / std > 0.05:
                errors.append(f"Price differs from standard: {name} ({price} vs {std})")

    return errors


# Test it on PO 4, using the JSON Claude gave you earlier
data_po4 = {
    "sender_email": "carlo@manilabaytrading.example",
    "distributor_as_written": "Manila Bay Trading",
    "po_number_subject": None,
    "po_number_body": None,
    "po_date": "2026-08-03",
    "delivery_date": "2026-08-31",
    "currency": "SGD",
    "items": [
        {"product": "Ultra Laundry Detergent 2kg", "quantity_raw": "240 pcs", "quantity_number": 240, "unit_price": 9.30},
        {"product": "Fabric Softener 1.5L", "quantity_raw": "two hundred pcs", "quantity_number": None, "unit_price": 5.53},
        {"product": "Lavender Floor Cleaner 1L", "quantity_raw": "150 pcs", "quantity_number": 150, "unit_price": 4.20},
    ],
}

result = validate(data_po4)
print("Errors found:", len(result))
for e in result:
    print("-", e)

Errors found: 2
- Missing PO number
- Quantity not a number: Fabric Softener 1.5L ('two hundred pcs')


In [11]:
import difflib
from datetime import date

def find_distributor(written):
    w = (written or "").lower().strip()
    if not w:
        return None, None
    names = registry["Distributor"].tolist()
    for n in names:
        if w == n.lower() or w in n.lower() or n.lower() in w:
            return n, "ok"
    close = difflib.get_close_matches(written, names, n=1, cutoff=0.85)
    if close:
        return close[0], "close"
    return None, None

def validate(data, seen):
    errors = []

    # PO number checks
    subj = data["po_number_subject"]
    body = data["po_number_body"]
    if not subj and not body:
        errors.append("Missing PO number")
    if subj and body and subj.strip() != body.strip():
        errors.append(f"Subject vs body PO number mismatch ({subj} vs {body})")
    po_no = body or subj
    if po_no:
        if po_no in seen:
            errors.append(f"Duplicate PO number: {po_no}")
        seen.add(po_no)

    # Distributor and sender checks
    written = data["distributor_as_written"]
    official, how = find_distributor(written)
    if official is None:
        errors.append(f"Unknown distributor: {written}")
    else:
        if how == "close":
            errors.append(f"Distributor name spelling variation: '{written}' vs '{official}'")
        official_email = registry.loc[registry["Distributor"] == official, "Official ordering email"].iloc[0]
        if (data["sender_email"] or "").strip().lower() != official_email.lower():
            errors.append(f"Sender email mismatch: {data['sender_email']} vs {official_email}")

    # Date check
    if data["po_date"] and data["delivery_date"]:
        if date.fromisoformat(data["delivery_date"]) < date.fromisoformat(data["po_date"]):
            errors.append("Delivery date before order date")

    # Item checks
    for item in data["items"]:
        name = item["product"]
        if item["quantity_number"] is None:
            errors.append(f"Quantity not a number: {name} ('{item['quantity_raw']}')")
        if name not in standard_prices:
            errors.append(f"Unknown product: {name}")
        else:
            std = standard_prices[name]
            price = item["unit_price"]
            if price is not None and abs(price - std) / std > 0.05:
                errors.append(f"Price differs from standard: {name} ({price} vs {std})")

    return errors


# A small helper to build test data quickly
def make(sender, name, subj, body, po_date, delivery, items):
    return {"sender_email": sender, "distributor_as_written": name,
            "po_number_subject": subj, "po_number_body": body,
            "po_date": po_date, "delivery_date": delivery, "currency": "SGD",
            "items": [{"product": p, "quantity_raw": str(q), "quantity_number": q, "unit_price": u}
                      for p, q, u in items]}

# Test POs typed from your answer key
po2 = make("orders@klangvalleydist.example", "Klang Valley Distribution",
           "KVD/2026/0315", "KVD/2026/0315", "2026-07-09", "2026-08-05",
           [("Electrolyte Sachets 12s", 400, 6.83), ("Vitamin C Gummies 60s", 250, 11.35)])
po3 = make("purchasing@siamretailpartners.example", "Siam Retail Partners Co Ltd",
           "SRP-26-0188", "SRP-26-0181", "2026-08-07", "2026-09-11",
           [("Collagen Drink Box 10s", 200, 14.5), ("Gentle Sunscreen SPF50 100ml", 300, 8.58)])
po7 = make("gudang@jakartaprima.example", "Jakarta Prima Distribusi PT",
           "JPD-2026-0233", "JPD-2026-0233", "2026-08-27", "2026-09-28",
           [("Sambal Chilli Sauce 250ml", 600, 3.34), ("Herbal Shampoo 400ml", 300, 4.77)])
po8 = make("purchasing@klangvalleydist.example", "Klang Valley Distribution",
           "KVD/2026/0315", "KVD/2026/0315", "2026-09-15", "2026-10-14",
           [("Herbal Shampoo 400ml", 300, 4.77), ("Aloe Fresh Body Wash 500ml", 350, 4.04)])

seen = set()
for label, po in [("PO 2", po2), ("PO 3", po3), ("PO 7", po7), ("PO 8", po8)]:
    print(label, "->", validate(po, seen))

PO 2 -> []
PO 3 -> ['Subject vs body PO number mismatch (SRP-26-0188 vs SRP-26-0181)', 'Price differs from standard: Collagen Drink Box 10s (14.5 vs 18.56)']
PO 7 -> ["Distributor name spelling variation: 'Jakarta Prima Distribusi PT' vs 'Jakarta Prima Distribution PT'"]
PO 8 -> ['Duplicate PO number: KVD/2026/0315', 'Sender email mismatch: purchasing@klangvalleydist.example vs orders@klangvalleydist.example']


In [12]:
from google.colab import files

instructions = """You are a data extraction tool for purchase order emails. Below are 8 separate
emails, each starting with a line like "=== EMAIL 1 ===". Treat every email on its own.
Return ONLY a valid JSON array with exactly 8 objects, one per email, in order.
No explanation, no markdown, no code fences. Use straight double quotes.

Rules:
- Copy values exactly as written. Do not correct spelling, prices or names.
- If a field is missing, use null. Never guess or invent a value.
- Do not fix errors and do not comment on them. Only extract.
- Dates must be in YYYY-MM-DD format.
- quantity_raw is the quantity exactly as written. quantity_number is the
  quantity as a number only if it is written in digits. If it is written in
  words, use null.
- po_number_subject is the PO number in the Subject line (null if none).
  po_number_body is the PO number written in the email body (null if none).
- po_date is the order date in the body if there is one. If there is none,
  use the email's Date line.

Format of each object:
{
  "email_no": 1,
  "sender_email": "",
  "distributor_as_written": "",
  "po_number_subject": null,
  "po_number_body": null,
  "po_date": "",
  "delivery_date": null,
  "currency": "",
  "items": [
    {"product": "", "quantity_raw": "", "quantity_number": null, "unit_price": null}
  ]
}
"""

body = ""
for p in pos:
    body += "\n=== EMAIL " + str(p["po_no"]) + " ===\n" + p["email"] + "\n"

full = instructions + body
open("claude_prompt.txt", "w", encoding="utf-8").write(full)
print(len(full), "characters")
files.download("claude_prompt.txt")

6025 characters


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [13]:
import json

raw = '''[{“email_no”:1,”sender_email”:”procurement@nusantaraniaga.example”,”distributor_as_written”:”Nusantara Niaga PT”,”po_number_subject”:”NN-PO-2026-0701",”po_number_body”:”NN-PO-2026-0701",”po_date”:”2026-07-02",”delivery_date”:”2026-07-30",”currency”:”SGD”,”items”:[{“product”:”Instant Kopi Mix 20s”,”quantity_raw”:”500",”quantity_number”:500,”unit_price”:6.06},{“product”:”Sambal Chilli Sauce 250ml”,”quantity_raw”:”300",”quantity_number”:300,”unit_price”:3.34},{“product”:”Coconut Biscuits 200g”,”quantity_raw”:”800",”quantity_number”:800,”unit_price”:2.10}]},{“email_no”:2,”sender_email”:”orders@klangvalleydist.example”,”distributor_as_written”:”Klang Valley Distribution”,”po_number_subject”:”KVD/2026/0315",”po_number_body”:”KVD/2026/0315",”po_date”:”2026-07-09",”delivery_date”:”2026-08-05",”currency”:”SGD”,”items”:[{“product”:”electrolyte sachets 12s”,”quantity_raw”:”400",”quantity_number”:400,”unit_price”:6.83},{“product”:”Vitamin C Gummies 60s”,”quantity_raw”:”250",”quantity_number”:250,”unit_price”:11.35},{“product”:”Herbal Throat Lozenges 24s”,”quantity_raw”:”600",”quantity_number”:600,”unit_price”:3.28},{“product”:”citrus dishwash liquid 750ml”,”quantity_raw”:”350",”quantity_number”:350,”unit_price”:2.98}]},{“email_no”:3,”sender_email”:”purchasing@siamretailpartners.example”,”distributor_as_written”:”Siam Retail Partners Co Ltd”,”po_number_subject”:”SRP-26-0188",”po_number_body”:”SRP-26-0188",”po_date”:”2026-07-21",”delivery_date”:”2026-08-25",”currency”:”SGD”,”items”:[{“product”:”Collagen Drink Box 10s”,”quantity_raw”:”200",”quantity_number”:200,”unit_price”:14.50},{“product”:”Gentle Sunscreen SPF50 100ml”,”quantity_raw”:”300",”quantity_number”:300,”unit_price”:8.58},{“product”:”Aloe Fresh Body Wash 500ml”,”quantity_raw”:”450",”quantity_number”:450,”unit_price”:4.04},{“product”:”Whitening Toothpaste 150g”,”quantity_raw”:”600",”quantity_number”:600,”unit_price”:2.49}]},{“email_no”:4,”sender_email”:”carlo@manilabaytrading.example”,”distributor_as_written”:”Manila Bay Trading”,”po_number_subject”:null,”po_number_body”:null,”po_date”:”2026-08-03",”delivery_date”:”2026-08-31",”currency”:”SGD”,”items”:[{“product”:”Ultra Laundry Detergent 2kg”,”quantity_raw”:”240 pcs”,”quantity_number”:240,”unit_price”:9.30},{“product”:”Fabric Softener 1.5L”,”quantity_raw”:”two hundred pcs”,”quantity_number”:null,”unit_price”:5.53},{“product”:”Lavender Floor Cleaner 1L”,”quantity_raw”:”150 pcs”,”quantity_number”:150,”unit_price”:4.20}]},{“email_no”:5,”sender_email”:”purchasing@saigonfresh.example”,”distributor_as_written”:”Saigon Fresh Distribution JSC”,”po_number_subject”:”SFD-2026-0412",”po_number_body”:”SFD-2026-0412",”po_date”:”2026-08-12",”delivery_date”:”2026-09-10",”currency”:”SGD”,”items”:[{“product”:”Jasmine Green Tea 25 bags”,”quantity_raw”:”700",”quantity_number”:700,”unit_price”:3.70},{“product”:”Matcha Latte Powder 15s”,”quantity_raw”:”300",”quantity_number”:300,”unit_price”:5.20},{“product”:”Herbal Shampoo 400ml”,”quantity_raw”:”280",”quantity_number”:280,”unit_price”:4.77}]},{“email_no”:6,”sender_email”:”ops@lioncitytrading.example”,”distributor_as_written”:”Lion City Trading Pte Ltd”,”po_number_subject”:”LCT-2026-0087",”po_number_body”:”LCT-2026-0087",”po_date”:”2026-08-18",”delivery_date”:”2026-08-15",”currency”:”SGD”,”items”:[{“product”:”Instant Kopi Mix 20s”,”quantity_raw”:”1000 units”,”quantity_number”:1000,”unit_price”:6.06},{“product”:”Coconut Biscuits 200g”,”quantity_raw”:”1200 units”,”quantity_number”:1200,”unit_price”:2.10}]},{“email_no”:7,”sender_email”:”gudang@jakartaprima.example”,”distributor_as_written”:”Jakarta Prima Distribusi PT”,”po_number_subject”:”JPD-2026-0233",”po_number_body”:”JPD-2026-0233",”po_date”:”2026-08-27",”delivery_date”:”2026-09-28",”currency”:”SGD”,”items”:[{“product”:”sambal chilli sauce 250ml”,”quantity_raw”:”600",”quantity_number”:600,”unit_price”:3.34},{“product”:”Ultra Laundry Detergent 2kg”,”quantity_raw”:”320",”quantity_number”:320,”unit_price”:9.30},{“product”:”Whitening Toothpaste 150g”,”quantity_raw”:”500pcs”,”quantity_number”:500,”unit_price”:2.49},{“product”:”jasmine green tea 25 bags”,”quantity_raw”:”400",”quantity_number”:400,”unit_price”:3.70},{“product”:”Herbal Shampoo 400ml”,”quantity_raw”:”qty300",”quantity_number”:300,”unit_price”:4.77}]},{“email_no”:8,”sender_email”:”purchasing@klangvalleydist.example”,”distributor_as_written”:”Klang Valley Distribution”,”po_number_subject”:”KVD/2026/0315",”po_number_body”:”KVD/2026/0315",”po_date”:”2026-09-15",”delivery_date”:”2026-10-14",”currency”:”SGD”,”items”:[{“product”:”Herbal Shampoo 400ml”,”quantity_raw”:”300 units”,”quantity_number”:300,”unit_price”:4.77},{“product”:”Aloe Fresh Body Wash 500ml”,”quantity_raw”:”350 units”,”quantity_number”:350,”unit_price”:4.04},{“product”:”Lavender Floor Cleaner 1L”,”quantity_raw”:”200 units”,”quantity_number”:200,”unit_price”:4.20},{“product”:”Fabric Softener 1.5L”,”quantity_raw”:”250 units”,”quantity_number”:250,”unit_price”:5.53}]}]'''
raw = raw.replace("“", '"').replace("”", '"')
extracted = json.loads(raw)
print("Emails extracted:", len(extracted))

# Cleanup: match product names to the price list, ignoring capitals and extra spaces
canonical = {name.lower(): name for name in standard_prices}
for d in extracted:
    for item in d["items"]:
        clean_name = " ".join(item["product"].lower().split())
        if clean_name in canonical:
            item["product"] = canonical[clean_name]

# Run all checks, in PO order
seen = set()
total = 0
for d in extracted:
    errs = validate(d, seen)
    total += len(errs)
    print("PO", d["email_no"], "->", len(errs), "error(s)")
    for e in errs:
        print("   -", e)
print("Total errors found:", total)

Emails extracted: 8
PO 1 -> 0 error(s)
PO 2 -> 0 error(s)
PO 3 -> 1 error(s)
   - Price differs from standard: Collagen Drink Box 10s (14.5 vs 18.56)
PO 4 -> 2 error(s)
   - Missing PO number
   - Quantity not a number: Fabric Softener 1.5L ('two hundred pcs')
PO 5 -> 1 error(s)
   - Unknown product: Matcha Latte Powder 15s
PO 6 -> 1 error(s)
   - Delivery date before order date
PO 7 -> 1 error(s)
   - Distributor name spelling variation: 'Jakarta Prima Distribusi PT' vs 'Jakarta Prima Distribution PT'
PO 8 -> 2 error(s)
   - Duplicate PO number: KVD/2026/0315
   - Sender email mismatch: purchasing@klangvalleydist.example vs orders@klangvalleydist.example
Total errors found: 8


In [14]:
print(pos[2]["email"])

From: purchasing@siamretailpartners.example
To: orders@sunbirdgoods.example
Date: 21 July 2026
Subject: Purchase Order SRP-26-0188


Dear Sunbird team,


Purchase Order: SRP-26-0188
Customer: Siam Retail Partners Co Ltd
Order date: 21 July 2026
Delivery date: 25 August 2026
Currency: SGD


+----+------------------------------+-----+-----------+
| No | Item                         | Qty | Unit Price|
+----+------------------------------+-----+-----------+
| 1  | Collagen Drink Box 10s       | 200 | 14.50     |
| 2  | Gentle Sunscreen SPF50 100ml | 300 | 8.58      |
| 3  | Aloe Fresh Body Wash 500ml   | 450 | 4.04      |
| 4  | Whitening Toothpaste 150g    | 600 | 2.49      |
+----+------------------------------+-----+-----------+


Best regards,
Somchai Wattana
Procurement, Siam Retail Partners Co Ltd


In [15]:
import re
print(re.findall(r"SRP-26-\d+", pos[2]["email"]))

['SRP-26-0188', 'SRP-26-0188']


In [16]:
import copy

expected = {
    1: [],
    2: [],
    3: ["Price differs from standard"],
    4: ["Missing PO number", "Quantity not a number"],
    5: ["Unknown product"],
    6: ["Delivery date before order date"],
    7: ["Distributor name spelling variation"],
    8: ["Duplicate PO number", "Sender email mismatch"],
}

seen = set()
caught = 0
missed = []
false_alarms = []

for d in extracted:
    n = d["email_no"]
    found = validate(d, seen)
    for exp in expected[n]:
        if any(f.startswith(exp) for f in found):
            caught += 1
        else:
            missed.append((n, exp))
    for f in found:
        if not any(f.startswith(exp) for exp in expected[n]):
            false_alarms.append((n, f))

total_expected = sum(len(v) for v in expected.values())
print(f"Caught {caught} of {total_expected} planted errors")
print("Missed:", missed)
print("False alarms:", false_alarms)

# Separate test of the subject-vs-body check, using a hand-made example
test = copy.deepcopy(extracted[2])
test["po_number_body"] = "SRP-26-0181"
print()
print("Hand-made mismatch test:")
for e in validate(test, set()):
    print("-", e)

Caught 8 of 8 planted errors
Missed: []
False alarms: []

Hand-made mismatch test:
- Subject vs body PO number mismatch (SRP-26-0188 vs SRP-26-0181)
- Price differs from standard: Collagen Drink Box 10s (14.5 vs 18.56)


In [17]:
import sqlite3
import os
import pandas as pd

if os.path.exists("orderflow.db"):
    os.remove("orderflow.db")

conn = sqlite3.connect("orderflow.db")
cur = conn.cursor()

cur.executescript("""
CREATE TABLE distributors (
    distributor_id INTEGER PRIMARY KEY,
    name TEXT,
    country TEXT,
    official_email TEXT
);
CREATE TABLE products (
    product_id INTEGER PRIMARY KEY,
    name TEXT,
    standard_price REAL
);
CREATE TABLE orders (
    order_id INTEGER PRIMARY KEY AUTOINCREMENT,
    po_number TEXT,
    po_date TEXT,
    delivery_date TEXT,
    sender_email TEXT,
    distributor_as_written TEXT,
    distributor_id INTEGER,
    status TEXT
);
CREATE TABLE order_items (
    item_id INTEGER PRIMARY KEY AUTOINCREMENT,
    order_id INTEGER,
    product TEXT,
    quantity_raw TEXT,
    quantity_number INTEGER,
    unit_price REAL,
    line_total REAL
);
CREATE TABLE exceptions (
    exception_id INTEGER PRIMARY KEY AUTOINCREMENT,
    order_id INTEGER,
    error_text TEXT
);
""")

# Fill the reference tables
for _, r in registry.iterrows():
    cur.execute("INSERT INTO distributors (name, country, official_email) VALUES (?, ?, ?)",
                (r["Distributor"], r["Country"], r["Official ordering email"]))
for name, price in standard_prices.items():
    cur.execute("INSERT INTO products (name, standard_price) VALUES (?, ?)", (name, price))

# Validate each PO and store it
seen = set()
for d in extracted:
    errs = validate(d, seen)

    official, how = find_distributor(d["distributor_as_written"])
    dist_id = None
    if official:
        dist_id = cur.execute("SELECT distributor_id FROM distributors WHERE name = ?",
                              (official,)).fetchone()[0]

    po_no = d["po_number_body"] or d["po_number_subject"]
    status = "clean" if not errs else "needs_review"

    cur.execute("""INSERT INTO orders (po_number, po_date, delivery_date, sender_email,
                   distributor_as_written, distributor_id, status)
                   VALUES (?, ?, ?, ?, ?, ?, ?)""",
                (po_no, d["po_date"], d["delivery_date"], d["sender_email"],
                 d["distributor_as_written"], dist_id, status))
    order_id = cur.lastrowid

    for item in d["items"]:
        qty = item["quantity_number"]
        price = item["unit_price"]
        line_total = round(qty * price, 2) if qty is not None and price is not None else None
        cur.execute("""INSERT INTO order_items (order_id, product, quantity_raw,
                       quantity_number, unit_price, line_total)
                       VALUES (?, ?, ?, ?, ?, ?)""",
                    (order_id, item["product"], item["quantity_raw"], qty, price, line_total))

    for e in errs:
        cur.execute("INSERT INTO exceptions (order_id, error_text) VALUES (?, ?)", (order_id, e))

conn.commit()

# Quick checks, written in SQL
print(pd.read_sql("SELECT status, COUNT(*) AS orders FROM orders GROUP BY status", conn))
print(pd.read_sql("SELECT COUNT(*) AS order_items FROM order_items", conn))
print(pd.read_sql("SELECT COUNT(*) AS exceptions FROM exceptions", conn))

         status  orders
0         clean       2
1  needs_review       6
   order_items
0           28
   exceptions
0           8


In [18]:
import pandas as pd

# Question 1: Which distributor sent each order, and what is its status?
q1 = """
SELECT o.order_id, o.po_number, d.name AS distributor, d.country, o.status
FROM orders o
LEFT JOIN distributors d ON o.distributor_id = d.distributor_id
ORDER BY o.order_id
"""

# Question 2: How many lines and what total value does each order have?
q2 = """
SELECT o.order_id, o.po_number, o.status,
       COUNT(i.item_id) AS lines,
       ROUND(SUM(i.line_total), 2) AS order_value
FROM orders o
JOIN order_items i ON i.order_id = o.order_id
GROUP BY o.order_id
ORDER BY order_value DESC
"""

# Question 3: What is in the review queue?
q3 = """
SELECT o.order_id, o.po_number, e.error_text
FROM exceptions e
JOIN orders o ON o.order_id = e.order_id
ORDER BY o.order_id
"""

for q in (q1, q2, q3):
    display(pd.read_sql(q, conn))

,order_id,po_number,distributor,country,status
0,1,NN-PO-2026-0701,Nusantara Niaga PT,Indonesia,clean
1,2,KVD/2026/0315,Klang Valley Distribution,Malaysia,clean
2,3,SRP-26-0188,Siam Retail Partners Co Ltd,Thailand,needs_review
3,4,None,Manila Bay Trading Corp,Philippines,needs_review
4,5,SFD-2026-0412,Saigon Fresh Distribution JSC,Vietnam,needs_review
5,6,LCT-2026-0087,Lion City Trading Pte Ltd,Singapore,needs_review
6,7,JPD-2026-0233,Jakarta Prima Distribution PT,Indonesia,needs_review
7,8,KVD/2026/0315,Klang Valley Distribution,Malaysia,needs_review


,order_id,po_number,status,lines,order_value
0,7,JPD-2026-0233,needs_review,5,9136.0
1,3,SRP-26-0188,needs_review,4,8786.0
2,2,KVD/2026/0315,clean,4,8580.5
3,6,LCT-2026-0087,needs_review,2,8580.0
4,1,NN-PO-2026-0701,clean,3,5712.0
5,5,SFD-2026-0412,needs_review,3,5485.6
6,8,KVD/2026/0315,needs_review,4,5067.5
7,4,None,needs_review,3,2862.0


,order_id,po_number,error_text
0,3,SRP-26-0188,Price differs from standard: Collagen Drink Bo...
1,4,None,Missing PO number
2,4,None,Quantity not a number: Fabric Softener 1.5L ('...
3,5,SFD-2026-0412,Unknown product: Matcha Latte Powder 15s
4,6,LCT-2026-0087,Delivery date before order date
5,7,JPD-2026-0233,Distributor name spelling variation: 'Jakarta ...
6,8,KVD/2026/0315,Duplicate PO number: KVD/2026/0315
7,8,KVD/2026/0315,Sender email mismatch: purchasing@klangvalleyd...
